In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pymupdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 47.4 MB/s eta 0:00:00


In [ ]:
import os
import re
import fitz

In [ ]:
pdf_folder = "/content/drive/MyDrive/PK-3/dataset"

output_folder = "/content/drive/MyDrive/PK-3/data/raw"

log_file = "/content/drive/MyDrive/PK-3/cleaning.log"

os.makedirs(output_folder, exist_ok=True)

In [ ]:
def clean_text(text):

    # =====================
    # HAPUS WATERMARK MA
    # =====================

    patterns = [
        r"^\s*hkama\s*$",
        r"^\s*blik indonesi\s*$",
        r".*agung repub.*",
        r".*republik indonesia.*",
        r".*direktori putusan.*",
        r".*mahagung\.go\.id.*"
    ]

    for pattern in patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )

    # =====================
    # HAPUS HEADER / FOOTER
    # =====================

    text = re.sub(
        r"Halaman\s+\d+\s+dari\s+\d+\s+Putusan\s+Nomor.*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Halaman\s+\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    # =====================
    # HAPUS KODE DOKUMEN
    # =====================

    text = re.sub(
        r"Pid\.[A-Z]\.[A-Z]\.\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    # =====================
    # HAPUS DISCLAIMER
    # =====================
    # Dinonaktifkan sementara karena
    # berpotensi menghapus isi putusan
    # dalam jumlah besar akibat re.DOTALL

    # text = re.sub(
    #     r"Disclaimer.*?Telp\s*:.*?(?:\n|$)",
    #     "",
    #     text,
    #     flags=re.IGNORECASE | re.DOTALL
    # )

    # text = re.sub(
    #     r"Disclaimer.*?Dalam hal Anda menemukan inakurasi.*?:",
    #     "",
    #     text,
    #     flags=re.IGNORECASE | re.DOTALL
    # )

    # text = re.sub(
    #     r"Disclaimer.*?(?=Setelah mendengar|Menimbang|DAKWAAN|$)",
    #     "",
    #     text,
    #     flags=re.IGNORECASE | re.DOTALL
    # )

    text = re.sub(
        r"^\s*Disclaimer\s*$",
        "",
        text,
        flags=re.IGNORECASE | re.MULTILINE
    )

    # =====================
    # RAPIIKAN SPASI
    # =====================

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r" *\n *", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
pdf_files = sorted([
    f for f in os.listdir(pdf_folder)
    if f.lower().endswith(".pdf")
])

print("Jumlah PDF:", len(pdf_files))

Jumlah PDF: 50


In [ ]:
def check_completeness(raw_text, cleaned_text):

    raw_words = len(raw_text.split())
    clean_words = len(cleaned_text.split())

    completeness = (
        clean_words / raw_words * 100
    ) if raw_words > 0 else 0

    status = (
        "VALID"
        if completeness >= 80
        else "PERLU CEK"
    )

    return {
        "raw_words": raw_words,
        "clean_words": clean_words,
        "completeness": completeness,
        "status": status
    }

In [ ]:
with open(log_file, "w", encoding="utf-8") as log:

    log.write("=== CLEANING LOG ===\n\n")

    for i, filename in enumerate(pdf_files, start=1):

        try:

            pdf_path = os.path.join(
                pdf_folder,
                filename
            )

            doc = fitz.open(pdf_path)

            all_text = []

            for page in doc:

                page_text = page.get_text("text")

                if page_text:
                    all_text.append(page_text)

            raw_text = "\n".join(all_text)

            cleaned_text = clean_text(raw_text)

            # VALIDASI KEUTUHAN TEKS
            validation = check_completeness(
                raw_text,
                cleaned_text
            )

            output_file = os.path.join(
                output_folder,
                f"case_{i:03d}.txt"
            )

            with open(
                output_file,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(cleaned_text)

            before_chars = len(raw_text)
            after_chars = len(cleaned_text)

            retention_ratio = (
                after_chars / before_chars
                if before_chars > 0
                else 0
            )

            log.write(
                f"{filename} -> case_{i:03d}.txt\n"
            )

            log.write(
                f"Before Characters : "
                f"{before_chars}\n"
            )

            log.write(
                f"After Characters : "
                f"{after_chars}\n"
            )

            log.write(
                f"Retention Ratio : "
                f"{retention_ratio:.4f}\n"
            )

            log.write(
                f"Raw Words : "
                f"{validation['raw_words']}\n"
            )

            log.write(
                f"Clean Words : "
                f"{validation['clean_words']}\n"
            )

            log.write(
                f"Text Completeness : "
                f"{validation['completeness']:.2f}%\n"
            )

            log.write(
                f"Status : "
                f"{validation['status']}\n\n"
            )

            print(
                f"✓ case_{i:03d}.txt | "
                f"ratio={retention_ratio:.4f} | "
                f"words={validation['clean_words']}/"
                f"{validation['raw_words']} | "
                f"completeness="
                f"{validation['completeness']:.2f}% | "
                f"{validation['status']}"
            )

        except Exception as e:

            print(
                f"✗ Gagal: {filename}"
            )

            print(e)

✓ case_001.txt | ratio=0.8205 | words=11015/12900 | completeness=85.39% | VALID
✓ case_002.txt | ratio=0.8019 | words=15822/18747 | completeness=84.40% | VALID
✓ case_003.txt | ratio=0.8392 | words=22423/25615 | completeness=87.54% | VALID
✓ case_004.txt | ratio=0.8077 | words=5466/6442 | completeness=84.85% | VALID
✓ case_005.txt | ratio=0.8018 | words=76660/90506 | completeness=84.70% | VALID
✓ case_006.txt | ratio=0.7973 | words=110394/131693 | completeness=83.83% | VALID
✓ case_007.txt | ratio=0.8029 | words=90920/108040 | completeness=84.15% | VALID
✓ case_008.txt | ratio=0.8073 | words=6847/7987 | completeness=85.73% | VALID
✓ case_009.txt | ratio=0.7583 | words=2774/3424 | completeness=81.02% | VALID
✓ case_010.txt | ratio=0.8195 | words=8696/10126 | completeness=85.88% | VALID
✓ case_011.txt | ratio=0.8310 | words=14251/16377 | completeness=87.02% | VALID
✓ case_012.txt | ratio=0.8310 | words=14251/16377 | completeness=87.02% | VALID
✓ case_013.txt | ratio=0.8099 | words=26903/

In [ ]:
txt_files = sorted([
    f for f in os.listdir(output_folder)
    if f.endswith(".txt")
])

print("Jumlah TXT:", len(txt_files))

Jumlah TXT: 50


In [ ]:
sample_file = os.path.join(
    output_folder,
    "case_033.txt"
)

with open(
    sample_file,
    "r",
    encoding="utf-8"
) as f:

    text = f.read()

print(text[:5000])

P U T U S A N
Nomor 489/Pid.B/2025/PN Bdg

DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA

Pengadilan Negeri Bandung yang mengadili perkara pidana dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara Terdakwa:
1. Nama lengkap
: Mahesa Putra Prayoga Bin (alm) Tatang Sutisna
2. Tempat lahir
: BANDUNG
3. Umur/Tanggal lahir
: 34/6 Juni 1991
4. Jenis kelamin
: Laki-laki
5. Kebangsaan
: Indonesia
6. Tempat tinggal
: Perum Puteraco Blok A1 No.14 Jl. Gg.Kina RT. 03
RW. 05 Kel. Pajajaran Kec. Cicendo Kota Bandung
7. Agama
: Islam
8. Pekerjaan
: Buruh harian lepas
Terdakwa Mahesa Putra Prayoga Bin (alm) Tatang Sutisna ditahan dalam
tahanan penyidik oleh:
1. Penyidik sejak tanggal 9 Mei 2025 sampai dengan tanggal 28 April 2025
2. Penyidik Perpanjangan Oleh Penuntut Umum sejak tanggal 29 April 2025
sampai dengan tanggal 7 Juni 2025
3. Penuntut Umum sejak tanggal 4 Juni 2025 sampai dengan tanggal 23 Juni
2025
4. Hakim Pengadilan Negeri sejak tang

In [ ]:
with open(
    log_file,
    "r",
    encoding="utf-8"
) as f:

    print(f.read()[:3000])

=== CLEANING LOG ===

putusan_1074_pid.b_2025_pn_bdg_20260615141736.pdf -> case_001.txt
Before Characters : 94458
After Characters : 77506
Retention Ratio : 0.8205
Raw Words : 12900
Clean Words : 11015
Text Completeness : 85.39%
Status : VALID

putusan_1091_pid.b_2025_pn_bdg_20260615145505.pdf -> case_002.txt
Before Characters : 139885
After Characters : 112175
Retention Ratio : 0.8019
Raw Words : 18747
Clean Words : 15822
Text Completeness : 84.40%
Status : VALID

putusan_1099_pid.b_2025_pn_bdg_20260615145419.pdf -> case_003.txt
Before Characters : 187795
After Characters : 157589
Retention Ratio : 0.8392
Raw Words : 25615
Clean Words : 22423
Text Completeness : 87.54%
Status : VALID

putusan_1100_pid.b_2024_pn_bdg_20260615140318.pdf -> case_004.txt
Before Characters : 46195
After Characters : 37311
Retention Ratio : 0.8077
Raw Words : 6442
Clean Words : 5466
Text Completeness : 84.85%
Status : VALID

putusan_1104_pid.b_2025_pn_bdg_20260615150004.pdf -> case_005.txt
Before Characters 